## 11 - Full Analysis on 75 Transcripts

This notebook re-runs sentiment scoring, hedge detection, and stock price
analysis on all 75 transcripts in our database.

### Why re-run?
Our original analysis only covered 14 transcripts.
Now that we have 75, we update all scores for the complete dataset.

### What we update?
1. Sentiment scores - VADER sentence by sentence scoring
2. Hedge detection - 42 word hedge dictionary
3. Stock price returns - 30, 60, 90 day returns after earnings
4. Master analysis table - everything combined

In [1]:
# Import all libraries
import sqlite3
import pandas as pd
import numpy as np
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from nltk.tokenize import sent_tokenize, word_tokenize
from collections import Counter

nltk.download('vader_lexicon')
nltk.download('punkt')
nltk.download('punkt_tab')

# Connect to database
conn = sqlite3.connect("../data/earnings_research.db")

# Load all 75 transcripts
transcripts = pd.read_sql("""
    SELECT id, ticker, quarter, date, period, raw_text
    FROM transcripts
    ORDER BY date
""", conn)

print(f"Loaded {len(transcripts)} transcripts")
print(transcripts[["ticker", "quarter", "period"]].to_string(index=False))

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\KOMAL\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\KOMAL\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\KOMAL\AppData\Roaming\nltk_data...


Loaded 75 transcripts
ticker quarter         period
  AAPL Q1_2020      Pre-COVID
   ABT Q1_2020      Pre-COVID
   AMD Q1_2020      Pre-COVID
   BAC Q1_2020      Pre-COVID
   BLK Q1_2020      Pre-COVID
  COST Q1_2020      Pre-COVID
   CRM Q1_2020      Pre-COVID
   CVX Q1_2020      Pre-COVID
   DHR Q1_2020      Pre-COVID
 GOOGL Q1_2020      Pre-COVID
    GS Q1_2020      Pre-COVID
   HAL Q1_2020      Pre-COVID
    HD Q1_2020      Pre-COVID
  INTC Q1_2020      Pre-COVID
    KO Q1_2020      Pre-COVID
   MCD Q1_2020      Pre-COVID
  META Q1_2020      Pre-COVID
   MRK Q1_2020      Pre-COVID
    MS Q1_2020      Pre-COVID
   NKE Q1_2020      Pre-COVID
   OXY Q1_2020      Pre-COVID
   PFE Q1_2020      Pre-COVID
    PG Q1_2020      Pre-COVID
  SBUX Q1_2020      Pre-COVID
   SLB Q1_2020      Pre-COVID
   TGT Q1_2020      Pre-COVID
   TMO Q1_2020      Pre-COVID
   UNH Q1_2020      Pre-COVID
   VLO Q1_2020      Pre-COVID
   WFC Q1_2020      Pre-COVID
   WMT Q1_2020      Pre-COVID
   JNJ Q1_2020    

[nltk_data]   Package punkt_tab is already up-to-date!


In [2]:
# Initialize VADER
sia = SentimentIntensityAnalyzer()

# Score all 75 transcripts sentence by sentence
sentiment_results = []

for _, row in transcripts.iterrows():
    sentences = sent_tokenize(row["raw_text"])
    
    sentence_scores = []
    for sentence in sentences:
        score = sia.polarity_scores(sentence)["compound"]
        sentence_scores.append(score)
    
    avg_compound = round(sum(sentence_scores) / len(sentence_scores), 4) if sentence_scores else 0
    max_positive = round(max(sentence_scores), 4) if sentence_scores else 0
    min_negative = round(min(sentence_scores), 4) if sentence_scores else 0
    
    sentiment_results.append({
        "ticker": row["ticker"],
        "quarter": row["quarter"],
        "date": row["date"],
        "period": row["period"],
        "avg_compound": avg_compound,
        "max_positive": max_positive,
        "min_negative": min_negative,
        "sentence_count": len(sentence_scores)
    })
    
    print(f"✓ {row['ticker']} {row['quarter']} → {avg_compound}")

print(f"\nTotal scored: {len(sentiment_results)}")

✓ AAPL Q1_2020 → 0.2968
✓ ABT Q1_2020 → 0.2587
✓ AMD Q1_2020 → 0.2125
✓ BAC Q1_2020 → 0.1907
✓ BLK Q1_2020 → 0.29
✓ COST Q1_2020 → 0.1049
✓ CRM Q1_2020 → 0.3347
✓ CVX Q1_2020 → 0.1566
✓ DHR Q1_2020 → 0.2963
✓ GOOGL Q1_2020 → 0.2679
✓ GS Q1_2020 → 0.2281
✓ HAL Q1_2020 → 0.1396
✓ HD Q1_2020 → 0.3021
✓ INTC Q1_2020 → 0.2545
✓ KO Q1_2020 → 0.2798
✓ MCD Q1_2020 → 0.1956
✓ META Q1_2020 → 0.227
✓ MRK Q1_2020 → 0.2381
✓ MS Q1_2020 → 0.181
✓ NKE Q1_2020 → 0.3747
✓ OXY Q1_2020 → 0.118
✓ PFE Q1_2020 → 0.2364
✓ PG Q1_2020 → 0.4284
✓ SBUX Q1_2020 → 0.4211
✓ SLB Q1_2020 → 0.1308
✓ TGT Q1_2020 → 0.243
✓ TMO Q1_2020 → 0.2611
✓ UNH Q1_2020 → 0.3288
✓ VLO Q1_2020 → 0.2443
✓ WFC Q1_2020 → 0.2561
✓ WMT Q1_2020 → 0.3882
✓ JNJ Q1_2020 → 0.2844
✓ JPM Q1_2020 → 0.1794
✓ MSFT Q1_2020 → 0.3113
✓ AAPL Q2_2020 → 0.3193
✓ AMZN Q1_2020 → 0.1272
✓ XOM Q1_2020 → 0.1838
✓ JPM Q2_2020 → 0.0901
✓ SLB Q2_2021 → 0.3493
✓ MSFT Q4_2022 → 0.2529
✓ AAPL Q4_2022 → 0.2228
✓ MSFT Q2_2023 → 0.2492
✓ ABT Q2_2023 → 0.2634
✓ AMD Q2_

In [3]:
# Save sentiment scores to database
df_sentiment = pd.DataFrame(sentiment_results)
df_sentiment.to_sql("sentiment_scores", conn, if_exists="replace", index=False)
conn.commit()
print(f"Sentiment scores saved for {len(df_sentiment)} transcripts")

# Now run hedge detection on all 75
hedge_words = [
    "may", "might", "could", "would", "should",
    "possibly", "perhaps", "potentially", "likely", "unlikely",
    "expect", "believe", "anticipate", "estimate", "assume",
    "suggest", "appear", "seem", "tend",
    "uncertainty", "risk", "challenge", "concern", "volatility",
    "headwind", "pressure", "difficulty", "weakness", "slowdown",
    "subject", "contingent", "dependent", "conditional",
    "approximately", "roughly", "around", "about",
    "forward", "outlook", "guidance", "forecast", "projection"
]

hedge_results = []

for _, row in transcripts.iterrows():
    words = word_tokenize(row["raw_text"].lower())
    total_words = len(words)
    found_hedges = [w for w in words if w in hedge_words]
    hedge_count = len(found_hedges)
    hedge_ratio = round(hedge_count / total_words, 4) if total_words > 0 else 0
    top_hedges = str(Counter(found_hedges).most_common(5))
    
    hedge_results.append({
        "ticker": row["ticker"],
        "quarter": row["quarter"],
        "date": row["date"],
        "period": row["period"],
        "total_words": total_words,
        "hedge_count": hedge_count,
        "hedge_ratio": hedge_ratio,
        "top_hedges": top_hedges
    })
    
    print(f"✓ {row['ticker']} {row['quarter']} → hedge ratio: {hedge_ratio}")

print(f"\nTotal analyzed: {len(hedge_results)}")

Sentiment scores saved for 75 transcripts
✓ AAPL Q1_2020 → hedge ratio: 0.0075
✓ ABT Q1_2020 → hedge ratio: 0.0105
✓ AMD Q1_2020 → hedge ratio: 0.0161
✓ BAC Q1_2020 → hedge ratio: 0.01
✓ BLK Q1_2020 → hedge ratio: 0.0121
✓ COST Q1_2020 → hedge ratio: 0.0134
✓ CRM Q1_2020 → hedge ratio: 0.0117
✓ CVX Q1_2020 → hedge ratio: 0.0163
✓ DHR Q1_2020 → hedge ratio: 0.0104
✓ GOOGL Q1_2020 → hedge ratio: 0.0093
✓ GS Q1_2020 → hedge ratio: 0.0167
✓ HAL Q1_2020 → hedge ratio: 0.0115
✓ HD Q1_2020 → hedge ratio: 0.0097
✓ INTC Q1_2020 → hedge ratio: 0.0093
✓ KO Q1_2020 → hedge ratio: 0.0146
✓ MCD Q1_2020 → hedge ratio: 0.0131
✓ META Q1_2020 → hedge ratio: 0.0127
✓ MRK Q1_2020 → hedge ratio: 0.0119
✓ MS Q1_2020 → hedge ratio: 0.0113
✓ NKE Q1_2020 → hedge ratio: 0.0095
✓ OXY Q1_2020 → hedge ratio: 0.0155
✓ PFE Q1_2020 → hedge ratio: 0.0161
✓ PG Q1_2020 → hedge ratio: 0.0109
✓ SBUX Q1_2020 → hedge ratio: 0.0074
✓ SLB Q1_2020 → hedge ratio: 0.0147
✓ TGT Q1_2020 → hedge ratio: 0.0143
✓ TMO Q1_2020 → hedge 

In [4]:
# Save hedge results to database
df_hedges = pd.DataFrame(hedge_results)
df_hedges.to_sql("hedge_analysis", conn, if_exists="replace", index=False)
conn.commit()
print(f"Hedge analysis saved for {len(df_hedges)} transcripts")

# Now run stock price analysis for all 75
def get_price_changes(ticker, earnings_date, conn):
    df_prices = pd.read_sql(f"""
        SELECT date, close
        FROM stock_prices
        WHERE ticker = '{ticker}'
        ORDER BY date
    """, conn)
    
    df_prices["date"] = pd.to_datetime(df_prices["date"].str[:10])
    earnings_date = pd.to_datetime(earnings_date)
    
    df_prices["days_diff"] = abs((df_prices["date"] - earnings_date).dt.days)
    earnings_row = df_prices.loc[df_prices["days_diff"].idxmin()]
    earnings_price = earnings_row["close"]
    
    results = {"earnings_price": round(earnings_price, 2)}
    
    for days in [30, 60, 90]:
        target_date = earnings_date + pd.Timedelta(days=days)
        df_prices["days_diff"] = abs((df_prices["date"] - target_date).dt.days)
        future_row = df_prices.loc[df_prices["days_diff"].idxmin()]
        future_price = future_row["close"]
        pct_change = round((future_price - earnings_price) / earnings_price * 100, 2)
        results[f"return_{days}d"] = pct_change
    
    return results

# Load sentiment for merging
df_sentiment = pd.read_sql("SELECT ticker, quarter, avg_compound FROM sentiment_scores", conn)

price_results = []

for _, row in df_hedges.iterrows():
    try:
        result = get_price_changes(row["ticker"], row["date"], conn)
        
        price_results.append({
            "ticker": row["ticker"],
            "quarter": row["quarter"],
            "date": row["date"],
            "period": row["period"],
            "hedge_ratio": row["hedge_ratio"],
            "earnings_price": result["earnings_price"],
            "return_30d": result["return_30d"],
            "return_60d": result["return_60d"],
            "return_90d": result["return_90d"]
        })
        
        print(f"✓ {row['ticker']} {row['quarter']} → 30d: {result['return_30d']}% | 90d: {result['return_90d']}%")
    
    except Exception as e:
        print(f"✗ {row['ticker']} {row['quarter']} failed: {e}")

print(f"\nTotal analyzed: {len(price_results)}")

Hedge analysis saved for 75 transcripts
✓ AAPL Q1_2020 → 30d: -13.7% | 90d: -10.65%
✓ ABT Q1_2020 → 30d: 17.84% | 90d: 19.91%
✓ AMD Q1_2020 → 30d: 14.25% | 90d: 20.5%
✓ BAC Q1_2020 → 30d: 16.74% | 90d: 20.97%
✓ BLK Q1_2020 → 30d: 18.22% | 90d: 33.74%
✓ COST Q1_2020 → 30d: 5.52% | 90d: 5.97%
✓ CRM Q1_2020 → 30d: 16.64% | 90d: 39.74%
✓ CVX Q1_2020 → 30d: 30.46% | 90d: 32.06%
✓ DHR Q1_2020 → 30d: 24.92% | 90d: 37.54%
✓ GOOGL Q1_2020 → 30d: 19.53% | 90d: 28.67%
✓ GS Q1_2020 → 30d: 21.89% | 90d: 36.87%
✓ HAL Q1_2020 → 30d: 47.88% | 90d: 97.4%
✓ HD Q1_2020 → 30d: 22.36% | 90d: 41.08%
✓ INTC Q1_2020 → 30d: 10.77% | 90d: 15.98%
✓ KO Q1_2020 → 30d: 8.26% | 90d: 7.04%
✓ MCD Q1_2020 → 30d: 15.48% | 90d: 17.41%
✓ META Q1_2020 → 30d: 26.74% | 90d: 42.27%
✓ MRK Q1_2020 → 30d: 5.24% | 90d: 5.62%
✓ MS Q1_2020 → 30d: 22.51% | 90d: 54.1%
✓ NKE Q1_2020 → 30d: 7.96% | 90d: 24.06%
✓ OXY Q1_2020 → 30d: 41.9% | 90d: 70.49%
✓ PFE Q1_2020 → 30d: 18.55% | 90d: 4.02%
✓ PG Q1_2020 → 30d: 7.56% | 90d: 10.1%
✓ SBUX

In [5]:
# Save price results and build master table
df_prices = pd.DataFrame(price_results)
df_prices.to_sql("price_analysis", conn, if_exists="replace", index=False)
conn.commit()
print("Price analysis saved!")

# Merge everything into master table
df_master = pd.merge(df_prices, df_sentiment, on=["ticker", "quarter"])
df_master["sentiment_hedge_gap"] = round(
    df_master["avg_compound"] - (df_master["hedge_ratio"] * 10), 4
)
df_master["performance_90d"] = df_master["return_90d"].apply(
    lambda x: "Outperformed" if x > 0 else "Underperformed"
)

df_master.to_sql("master_analysis", conn, if_exists="replace", index=False)
conn.commit()

print(f"Master table saved with {len(df_master)} records!")
print("\nTop 10 by 90-day return:")
print(df_master.nlargest(10, "return_90d")[["ticker", "quarter", "period", "avg_compound", "hedge_ratio", "return_90d"]].to_string(index=False))
print("\nBottom 10 by 90-day return:")
print(df_master.nsmallest(10, "return_90d")[["ticker", "quarter", "period", "avg_compound", "hedge_ratio", "return_90d"]].to_string(index=False))

Price analysis saved!
Master table saved with 75 records!

Top 10 by 90-day return:
ticker quarter    period  avg_compound  hedge_ratio  return_90d
   HAL Q1_2020 Pre-COVID        0.1396       0.0115       97.40
   OXY Q1_2020 Pre-COVID        0.1180       0.0155       70.49
    MS Q1_2020 Pre-COVID        0.1810       0.0113       54.10
   SLB Q1_2020 Pre-COVID        0.1308       0.0147       47.07
   VLO Q1_2020 Pre-COVID        0.2443       0.0151       45.72
  META Q1_2020 Pre-COVID        0.2270       0.0127       42.27
    HD Q1_2020 Pre-COVID        0.3021       0.0097       41.08
   CRM Q1_2020 Pre-COVID        0.3347       0.0117       39.74
   DHR Q1_2020 Pre-COVID        0.2963       0.0104       37.54
    GS Q1_2020 Pre-COVID        0.2281       0.0167       36.87

Bottom 10 by 90-day return:
ticker quarter  period  avg_compound  hedge_ratio  return_90d
    MS Q2_2023 AI Boom        0.2157       0.0149      -21.43
   TMO Q2_2023 AI Boom        0.2589       0.0169      -20.

In [6]:
conn.close()
print("Database connection closed.")

Database connection closed.
